# SINet — Camouflaged Object Detection: Inference on Images & Video
EE5340 Video Analytics, IIT Guwahati (Group project). Runtime → Change runtime type → **T4 GPU**, then Run all.

In [ ]:
!git clone https://github.com/akashshukla-github/SINet-COD-VideoAnalytics.git
%cd SINet-COD-VideoAnalytics/code
!pip -q install gdown opencv-python-headless

## 1. Download pretrained SINet weights + test set (links from the official SINet README)
If `gdown` fails (Drive quota), download manually from the README links and upload via the Files panel.

In [ ]:
!gdown 1iCkHXU55Or4gEtSqesijea6A5r8lVEYv -O SINet_40.pth || echo 'Manual download needed'
!gdown 1QEGnP9O7HbN_2tH999O3HRIsErIVYalx -O TestDataset.zip || echo 'Manual download needed'
!mkdir -p Dataset && unzip -q -o TestDataset.zip -d Dataset/
!find Dataset -maxdepth 3 -type d | head

## 2. Image results on COD10K-test (writes `../results/` used by the webpage)
Set `--data_root` to the folder that contains `COD10K/Image` and `COD10K/GT` (check the `find` output above).

In [ ]:
!python test_images.py --model_path SINet_40.pth --data_root ./Dataset/TestDataset --dataset COD10K --limit 300 --panels 4

## 3. Video inference (our Video Analytics extension)
Upload any camouflaged-animal clip as `input.mp4` (Files panel).

In [ ]:
!python video_inference.py --video input.mp4 --model_path SINet_40.pth --out_dir ../results/video --alpha 0.6 --tau 0.5 --max_frames 300
!ls ../results ../results/video

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('../results/video/track.csv')
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(df.frame, df.area_px); ax[0].set_title('Detected area (px) vs frame'); ax[0].set_xlabel('frame')
ax[1].plot(df.frame, pd.to_numeric(df.disp_px, errors='coerce')); ax[1].set_title('Centroid displacement (px/frame)'); ax[1].set_xlabel('frame')
plt.tight_layout(); plt.savefig('../results/video_track_plot.png', dpi=130); plt.show()
print('Detection rate:', df.detected.mean())

In [ ]:
# Copy a few video frames for the report
import glob, shutil
for i, f in enumerate(sorted(glob.glob('../results/video/frame_*.png'))[:3]):
    shutil.copy(f, f'../results/video_frame_{i+1}.png')
!cd .. && zip -r results.zip results
from google.colab import files; files.download('../results.zip')